In [1]:
!pip install implicit -U -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10793386 sha256=c73419719963f20dad3afe0f263ceed3b17ba4fc9400eef5a2a5cda96c6c7ba7
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [245]:
import polars as pl
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from implicit.nearest_neighbours import CosineRecommender, BM25Recommender,TFIDFRecommender
from implicit.bpr import BayesianPersonalizedRanking
from implicit.als import AlternatingLeastSquares
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
import torch
import scipy.sparse as sp
import gc
from collections import defaultdict, Counter
import math, os, random

In [246]:
def load_data(days_to_test=7):
    ds = load_dataset("yandex/yambda", data_dir="flat/5b", data_files="likes.parquet")
    data = ds['train'].to_polars().sort(['uid','timestamp'])
    data = data.with_columns(
        (pl.col("timestamp") / 86_400).cast(int).alias("days_from_start")
    )
    train_data = data.filter(
        pl.col('days_from_start') <= 300 - days_to_test
    )

    test_data = data.filter(
        pl.col('days_from_start') > 300 - days_to_test
    )
    
    del data
    gc.collect()

    train_users = train_data['uid'].unique().to_list()
    test_data = test_data.filter(
        pl.col('uid').is_in(train_users)
    )
    test_users = test_data['uid'].unique()
    test_data = test_data.group_by('uid').agg(
        pl.col('item_id').alias('items')
    )
    return train_data, test_data, test_users

In [ ]:
def _dedup_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def evaluate(
    gt_df,
    pred_df,
    model_name="none",
    ks=[10, 50, 100],
    uid_col="uid",
    items_col="items",
    eval_on="gt",
    skip_empty_gt=True,
    dedup_pred=True,
):
    ks = sorted(set(int(k) for k in ks))
    max_k = max(ks)

    gt_map = {}
    for row in gt_df.select([uid_col, items_col]).iter_rows(named=True):
        uid = row[uid_col]
        items = row[items_col] if row[items_col] is not None else []
        gt_map[uid] = list(items)

    pred_map = {}
    for row in pred_df.select([uid_col, items_col]).iter_rows(named=True):
        uid = row[uid_col]
        items = row[items_col] if row[items_col] is not None else []
        pred_map[uid] = list(items)

    if eval_on == "gt":
        uids = list(gt_map.keys())
    elif eval_on == "union":
        uids = list(set(gt_map.keys()) | set(pred_map.keys()))

    sum_recall = {k: 0.0 for k in ks}
    sum_ndcg = {k: 0.0 for k in ks}
    n_users = 0

    inv_logs = [1.0 / math.log2(i + 2) for i in range(max_k)]

    for uid in uids:
        gt_items = gt_map.get(uid, [])
        gt_set = set(gt_items)

        if len(gt_set) == 0:
            if skip_empty_gt:
                continue
            for k in ks:
                sum_recall[k] += 0.0
                sum_ndcg[k] += 0.0
            n_users += 1
            continue

        pred_items = pred_map.get(uid, [])
        if dedup_pred:
            pred_items = _dedup_preserve_order(pred_items)
        pred_top = pred_items[:max_k]

        cum_hits = []
        cum_dcg = []
        hits = 0
        dcg = 0.0
        for i, item in enumerate(pred_top):
            rel = 1 if item in gt_set else 0
            hits += rel
            dcg += rel * inv_logs[i]
            cum_hits.append(hits)
            cum_dcg.append(dcg)

        def _get_prefix(arr, k):
            idx = min(k, len(arr)) - 1
            return arr[idx] if idx >= 0 else 0

        for k in ks:
            hit_k = _get_prefix(cum_hits, k)
            dcg_k = float(_get_prefix(cum_dcg, k))

            recall_k = hit_k / len(gt_set)

            ideal_len = min(k, len(gt_set))
            idcg_k = sum(inv_logs[i] for i in range(ideal_len))
            ndcg_k = (dcg_k / idcg_k) if idcg_k > 0 else 0.0

            sum_recall[k] += recall_k
            sum_ndcg[k] += ndcg_k

        n_users += 1

    row = {"model": model_name, "users": n_users}
    for k in ks:
        row[f"recall@{k}"] = (sum_recall[k] / n_users) if n_users else 0.0
        row[f"ndcg@{k}"] = (sum_ndcg[k] / n_users) if n_users else 0.0

    cols = ["model", "users"] + [f"recall@{k}" for k in ks] + [f"ndcg@{k}" for k in ks]
    return pl.DataFrame([row]).select(cols)

In [248]:
def set_random_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(56)

In [249]:
train_data, test_data, test_users = load_data()

## Random

In [250]:
def get_random_predict(train_data, test_users, k=100):
    train_items = train_data['item_id'].unique().to_list()
    preds = np.random.choice(train_items,size=(len(test_users),k))
    return pl.DataFrame({'uid':test_users, 'items': preds})

In [251]:
random_preds = get_random_predict(train_data, test_users)
random_metrics = evaluate(test_data, random_preds, model_name='RandomModel')
random_metrics

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""RandomModel""",389065,0.000005,0.000026,0.00005,0.000006,0.000012,0.000018


## TopPopular

In [252]:
def generate_top_popular_predict(train_data, test_users, cutoff=7, k=100):
    top_items = train_data.filter(
        (pl.col('days_from_start') > pl.col('days_from_start').max() - cutoff)
    ).group_by('item_id').count()
    top_items = top_items.sort('count', descending=True)['item_id'][:k].to_list()
    return pl.DataFrame({'uid':test_users, 'items': [top_items] * len(test_users)})

In [253]:
toppop_preds = generate_top_popular_predict(train_data, test_users,cutoff=1000)
toppop_metrics = evaluate(test_data, toppop_preds, model_name='PopularModel')
toppop_metrics

/tmp/ipykernel_106/3995999292.py:4: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  ).group_by('item_id').count()


model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""PopularModel""",389065,0.004919,0.018403,0.030011,0.003791,0.007708,0.010529


In [254]:
toppop_preds = generate_top_popular_predict(train_data, test_users,cutoff=7)
toppop_metricsV2 = evaluate(test_data, toppop_preds, model_name='PopularModelTLastWeek')
toppop_metricsV2

/tmp/ipykernel_106/3995999292.py:4: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  ).group_by('item_id').count()


model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""PopularModelTLastWeek""",389065,0.022581,0.049739,0.07034,0.016618,0.023979,0.02895


## HistoryLookUp

In [255]:
def generate_history_look_up_predict(train_data, test_users, k=100):
    preds = train_data.filter(
        pl.col('uid').is_in(test_users)
    ).group_by('uid').agg(
        pl.col('item_id').alias('items')
    ).with_columns(
        pl.col('items').list.tail(k)
    )
    return preds

In [256]:
history_preds = generate_history_look_up_predict(train_data, test_users)
history_metrics = evaluate(test_data, history_preds, model_name='HitoryModel')
history_metrics

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""HitoryModel""",389065,0.004254,0.014015,0.024074,0.003203,0.005942,0.008342


## Item KNN & ALS

In [257]:
def get_coo_matrix(
    df,
    user_col='user_id',
    item_col='item_id',
    weight_col=None,
    users_mapping=None,
    items_mapping=None
):
    if weight_col is None:
        weights = np.ones(len(df), dtype=np.float32)
    else:
        weights = df[weight_col].to_pandas().astype(np.float32)

    interaction_matrix = sp.coo_matrix((
        weights,
        (
            df[user_col].to_pandas().map(users_mapping.get),
            df[item_col].to_pandas().map(items_mapping.get)
        )),
    )
    return interaction_matrix

In [258]:
def predict_impl(model,test_users,mat,users_mapping,items_inv_mapping,N=20,falh=True):
    recs,scores = [],[]
    for id in tqdm(test_users):
        row_id = users_mapping[id]
        ranks = model.recommend(row_id, mat[row_id], N=N, filter_already_liked_items=falh)
        recs += [[items_inv_mapping.get(it) for it in ranks[0]]]
        scores += [ranks[1]]
    return recs,scores

In [259]:
users_inv_mapping = dict(enumerate(train_data['uid'].unique()))
users_mapping = {v: k for k, v in users_inv_mapping.items()}

items_inv_mapping = dict(enumerate(train_data['item_id'].unique()))
items_mapping = {v: k for k, v in items_inv_mapping.items()}
len(users_mapping),len(items_mapping)

(820637, 1977318)

In [265]:
train_mat = get_coo_matrix(
    df=train_data,
    user_col='uid',
    item_col='item_id',
    users_mapping=users_mapping,
    items_mapping=items_mapping
).tocsr()

In [266]:
train_mat_time = get_coo_matrix(
    df=train_data,
    user_col='uid',
    item_col='item_id',
    weight_col='days_from_start',
    users_mapping=users_mapping,
    items_mapping=items_mapping
).tocsr()

In [267]:
model_tfidf = TFIDFRecommender(K=128)
model_tfidf.fit(train_mat)

recs, _ = predict_impl(model_tfidf, test_users, train_mat, users_mapping, items_inv_mapping, N=100)
tfidf_preds = pl.DataFrame({'uid':test_users, 'items':recs})
tfidf_metrics = evaluate(test_data, tfidf_preds, model_name='TfIDFModel')
tfidf_metrics

/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.5033659934997559 seconds
  warnings.warn(


  0%|          | 0/1977318 [00:00<?, ?it/s]

  0%|          | 0/389065 [00:00<?, ?it/s]

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""TfIDFModel""",389065,0.011669,0.042007,0.070066,0.010531,0.019176,0.026262


In [268]:
model_tfidf = TFIDFRecommender(K=128)
model_tfidf.fit(train_mat_time)

recs, _ = predict_impl(model_tfidf, test_users, train_mat_time, users_mapping, items_inv_mapping, N=100)

tfidf_preds = pl.DataFrame({'uid':test_users, 'items':recs})
tfidf_metricsV2 = evaluate(test_data, tfidf_preds, model_name='TfIDFModelTimeW')
tfidf_metricsV2

/usr/local/lib/python3.12/dist-packages/implicit/nearest_neighbours.py:233: RuntimeWarning: invalid value encountered in divide
  X.data = X.data / sqrt(bincount(X.row, X.data**2))[X.row]
/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.5042459964752197 seconds
  warnings.warn(


  0%|          | 0/1977318 [00:00<?, ?it/s]

  0%|          | 0/389065 [00:00<?, ?it/s]

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""TfIDFModelTimeW""",389065,0.014604,0.049539,0.079824,0.012847,0.022828,0.030493


In [269]:
model_als = AlternatingLeastSquares(
    factors = 256,
    iterations = 32,
    calculate_training_loss = False,
    regularization = 0.1
)

model_als.fit(train_mat)

recs, _ = predict_impl(model_als, test_users, train_mat, users_mapping, items_inv_mapping, N=100)

als_preds = pl.DataFrame({'uid':test_users, 'items':recs})
als_metrics = evaluate(test_data, als_preds, model_name='ALSModel')
als_metrics

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/389065 [00:00<?, ?it/s]

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""ALSModel""",389065,0.012493,0.045677,0.076358,0.011408,0.020914,0.028719


In [270]:
model_als = AlternatingLeastSquares(
    factors = 256,
    iterations = 32,
    calculate_training_loss = False,
    regularization = 0.1
)

model_als.fit(train_mat_time)

recs, _ = predict_impl(model_als, test_users, train_mat_time, users_mapping, items_inv_mapping, N=100)

als_preds = pl.DataFrame({'uid':test_users, 'items':recs})
als_metricsV2 = evaluate(test_data, als_preds, model_name='ALSModelTimeW')
als_metricsV2

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/389065 [00:00<?, ?it/s]

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""ALSModelTimeW""",389065,0.015687,0.057305,0.096977,0.01321,0.025383,0.035361


## Markov Model

In [271]:
class MarkovChainTopNext:
    def __init__(self, topk=100):
        self.topk = topk
        self.next_counts = defaultdict(Counter)
        self.top_next = {}

    def fit(self, sequences):
        nc = self.next_counts
        for seq in tqdm(sequences):
            if seq is None:
                continue
            n = len(seq)
            if n < 2:
                continue
            for a, b in zip(seq[:-1], seq[1:]):
                nc[a][b] += 1

        top_next = {}
        for a, counter in nc.items():
            items = list(counter.items())
            items.sort(key=lambda x: (-x[1], str(x[0])))
            top_next[a] = [x[0] for x in items[: self.topk]]

        self.top_next = top_next
        return self

    def predict_next_batch(self, items, k=None):
        if k is None:
            k = self.topk
        return [self.top_next.get(it, [])[:k] for it in items]

In [272]:
train_seqs = train_data.group_by('uid').agg(
    pl.col('item_id').alias('item_seq')
)['item_seq'].to_list()

In [273]:
model = MarkovChainTopNext()
model.fit(train_seqs)

  0%|          | 0/820637 [00:00<?, ?it/s]

In [274]:
test_last = train_data.filter(
    pl.col('uid').is_in(test_users)
).group_by('uid').agg(
    pl.col('item_id')
).with_columns(
    pl.col('item_id').list.tail(1)
)
test_last = [x[0] for x in test_last['item_id'].to_list()]
preds = model.predict_next_batch(test_last)

markov_preds = pl.DataFrame({'uid':test_users, 'items': preds})
markov_metrics = evaluate(test_data, markov_preds, model_name='MarkovChain')
markov_metrics

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""MarkovChain""",389065,0.016378,0.039382,0.055785,0.013933,0.020234,0.024243


## Total DataFrame

In [278]:
results = pl.concat([
    random_metrics,
    toppop_metrics,
    toppop_metricsV2,
    tfidf_metrics,
    tfidf_metricsV2,
    als_metrics,
    als_metricsV2,
    markov_metrics,
])
results

model,users,recall@10,recall@50,recall@100,ndcg@10,ndcg@50,ndcg@100
str,i64,f64,f64,f64,f64,f64,f64
"""RandomModel""",389065,0.000005,0.000026,0.00005,0.000006,0.000012,0.000018
"""PopularModel""",389065,0.004919,0.018403,0.030011,0.003791,0.007708,0.010529
"""PopularModelTLastWeek""",389065,0.022581,0.049739,0.07034,0.016618,0.023979,0.02895
"""TfIDFModel""",389065,0.011669,0.042007,0.070066,0.010531,0.019176,0.026262
"""TfIDFModelTimeW""",389065,0.014604,0.049539,0.079824,0.012847,0.022828,0.030493
"""ALSModel""",389065,0.012493,0.045677,0.076358,0.011408,0.020914,0.028719
"""ALSModelTimeW""",389065,0.015687,0.057305,0.096977,0.01321,0.025383,0.035361
"""MarkovChain""",389065,0.016378,0.039382,0.055785,0.013933,0.020234,0.024243
